# Detecção de Objetos via WhatsApp com YOLOv8

Pipeline completo:
1. Recebe foto via WhatsApp (webhook Twilio + ngrok)
2. Analisa a imagem com YOLOv8 para detectar qualquer objeto (animais, pessoas, veículos, etc.)
3. Envia relatório com todas as detecções e confiança de volta pelo WhatsApp

In [ ]:
!pip install ultralytics flask pyngrok requests pillow

In [ ]:
# ============================================================
# Definição de Variáveis
# ============================================================

TWILIO_ACCOUNT_SID = "YOUR_TWILIO_ACCOUNT_SID"
TWILIO_AUTH_TOKEN = ""
TWILIO_WHATSAPP_FROM = "whatsapp:+14155238886"

TWILIO_MESSAGES_URL = f"https://api.twilio.com/2010-04-01/Accounts/{TWILIO_ACCOUNT_SID}/Messages.json"

YOLO_MODEL_NAME = "yolov8n.pt"

FLASK_PORT = 5000
NGROK_AUTH_TOKEN = ""  # preencha com seu token do ngrok (https://dashboard.ngrok.com)

print("Variaveis definidas!")

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO(YOLO_MODEL_NAME)

print(f"Modelo {YOLO_MODEL_NAME} carregado!")
print(f"Total de classes detectaveis: {len(yolo_model.names)}")

In [ ]:
import requests
import tempfile
from PIL import Image


def baixar_imagem(media_url):
    """Baixa a imagem do Twilio usando autenticação básica."""
    resp = requests.get(media_url, auth=(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN))
    resp.raise_for_status()
    tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
    tmp.write(resp.content)
    tmp.close()
    return tmp.name


def analisar_imagem(caminho_imagem):
    """Executa YOLOv8 e retorna todas as detecções sem filtro de classe."""
    results = yolo_model(caminho_imagem, verbose=False)

    deteccoes = []
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            deteccoes.append({
                "nome": yolo_model.names[cls_id],
                "confianca": float(box.conf[0]),
            })

    return deteccoes


def gerar_relatorio(deteccoes):
    """Gera texto do relatório a partir das detecções."""
    if not deteccoes:
        return "Nenhum objeto foi detectado na imagem."

    contagem = {}
    for d in deteccoes:
        nome = d["nome"]
        if nome not in contagem:
            contagem[nome] = {"qty": 0, "max_conf": 0.0}
        contagem[nome]["qty"] += 1
        contagem[nome]["max_conf"] = max(contagem[nome]["max_conf"], d["confianca"])

    linhas = ["Relatorio de Deteccao"]
    linhas.append(f"Total de objetos detectados: {len(deteccoes)}")
    linhas.append("---")
    for nome, info in sorted(contagem.items(), key=lambda x: -x[1]["qty"]):
        linhas.append(
            f"- {nome.capitalize()}: {info['qty']}x "
            f"(confianca max: {info['max_conf']*100:.1f}%)"
        )

    return "\n".join(linhas)


def enviar_whatsapp(destino, mensagem):
    """Envia mensagem de texto via Twilio WhatsApp API."""
    resp = requests.post(
        TWILIO_MESSAGES_URL,
        data={
            "To": destino,
            "From": TWILIO_WHATSAPP_FROM,
            "Body": mensagem,
        },
        auth=(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN),
    )
    return resp.status_code, resp.json()


print("Funcoes auxiliares prontas!")

In [ ]:
import os
import threading
from flask import Flask, request

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():
    num_media = int(request.form.get("NumMedia", 0))
    remetente = request.form.get("From", "")
    profile = request.form.get("ProfileName", "Desconhecido")

    print(f"Mensagem recebida de {profile} ({remetente}) | Midias: {num_media}")

    if num_media == 0:
        enviar_whatsapp(remetente, "Envie uma foto para que eu possa analisar os animais!")
        return "OK", 200

    content_type = request.form.get("MediaContentType0", "")
    if not content_type.startswith("image/"):
        enviar_whatsapp(remetente, "Por favor, envie uma imagem (JPEG/PNG).")
        return "OK", 200

    media_url = request.form.get("MediaUrl0", "")
    print(f"Baixando imagem: {media_url}")

    try:
        caminho = baixar_imagem(media_url)
        print(f"Imagem salva em: {caminho}")

        deteccoes = analisar_imagem(caminho)
        print(f"Deteccoes: {deteccoes}")

        relatorio = gerar_relatorio(deteccoes)
        print(f"Relatorio:\n{relatorio}")

        status, resp = enviar_whatsapp(remetente, relatorio)
        print(f"Resposta enviada (status {status})")

        os.remove(caminho)
    except Exception as e:
        print(f"Erro ao processar imagem: {e}")
        enviar_whatsapp(remetente, f"Erro ao processar a imagem: {e}")

    return "OK", 200


print("Servidor Flask definido! Rota POST /webhook pronta.")

In [ ]:
from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN

public_url = ngrok.connect(FLASK_PORT)
webhook_url = f"{public_url}/webhook"

print("=" * 60)
print(f"URL publica do ngrok: {public_url}")
print(f"Webhook URL: {webhook_url}")
print("=" * 60)
print("Configure esta URL no Twilio Sandbox:")
print("  https://console.twilio.com/us1/develop/sms/try-it-out/whatsapp-learn")
print(f"  Coloque '{webhook_url}' em 'When a message comes in'")
print("=" * 60)

thread = threading.Thread(target=lambda: app.run(port=FLASK_PORT, use_reloader=False))
thread.daemon = True
thread.start()

print(f"Servidor rodando na porta {FLASK_PORT}. Envie uma foto no WhatsApp!")